# 01. Ingest and Deduplicate

This notebook walks through one document end to end with the current `GraphRAG` step API: read, chunk, embed, persist, extract entities, inspect the graph, search it, find duplicate candidates, merge duplicates, and query the cleaned graph again.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --group falkordb --extra notebooks
```

You also need model credentials for the chat and embedding models you choose below.


In [ ]:
import os
from pathlib import Path

import autoroot  # noqa
from dotenv import load_dotenv
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv()


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DATA_DIR = NOTEBOOKS_DIR / "experimental_data"
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
SOURCE_PATH = DATA_DIR / "deep_knowledge_from_deep_learning.txt"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter:openai/gpt-5-mini"
)
EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

In [2]:
database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(
    model=MODEL,
    embedding_model=EMBEDDING_MODEL,
    db=database,
    resolve_entities_on_ingest=True,
)

print("GraphRAG initialized with ingest-time entity resolution enabled.")

GraphRAG initialized with ingest-time entity resolution enabled.

## Step-by-step ingestion

The following cell uses the public notebook-friendly methods directly instead of the one-shot `ingest(...)` wrapper. That makes every intermediate artifact visible.


In [3]:
document = rag.read_document(SOURCE_PATH)
chunks = rag.chunk_document(document)
document_embedding = await rag.embed_document(document)
chunk_embeddings = await rag.embed_chunks(chunks)
document_node = rag.build_document_node(document, document_embedding)
chunk_nodes = rag.build_chunk_nodes(chunks, chunk_embeddings)
await rag.persist_document_and_chunks(document_node, chunk_nodes)
chunk_graphs = await rag.extract_kg_per_chunk(chunks)
await rag.persist_entities_and_relationships(
    [chunk.id for chunk in chunks], chunk_graphs
)

print(
    {
        "document_id": document.id,
        "chunk_count": len(chunks),
        "document_embedding_dims": len(document_embedding),
        "chunk_graph_count": len(chunk_graphs),
    }
)

{
    'document_id': '92ab22bd-3e12-4230-b1c4-2c762b522370',
    'chunk_count': 5,
    'document_embedding_dims': 1536,
    'chunk_graph_count': 5
}

In [4]:
document_rows = database.ro_query(
    "MATCH (n:__document__) RETURN n.id, n.name ORDER BY n.name LIMIT 3"
).result_set
chunk_rows = database.ro_query(
    "MATCH (n:__chunk__) RETURN n.id, n.name, n.document_id ORDER BY n.id LIMIT 5"
).result_set
entity_rows = database.ro_query(
    "MATCH (n:__entity__) RETURN n.id, n.name, n.semantic_key ORDER BY n.semantic_key, n.id LIMIT 10"
).result_set
entities = await database.list_entities()

print({"documents": document_rows, "chunks": chunk_rows, "entity_count": len(entities)})
print("First persisted entities:")
for entity in entities[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

{
    'documents': [['32809404-120d-4cae-a3b6-cad3e7ccce13', 'deep_knowledge_from_deep_learning']],
    'chunks': [
        [
            '03a40b69-0fd4-4a75-9be9-ea9cf16acc09',
            'Chunk 03a40b69-0fd4-4a75-9be9-ea9cf16acc09',
            '32809404-120d-4cae-a3b6-cad3e7ccce13'
        ],
        [
            '0f65886d-42fc-4596-932c-f28822e09d32',
            'Chunk 0f65886d-42fc-4596-932c-f28822e09d32',
            '32809404-120d-4cae-a3b6-cad3e7ccce13'
        ],
        [
            '11aefb96-5516-440f-ad73-bc86064102ea',
            'Chunk 11aefb96-5516-440f-ad73-bc86064102ea',
            '32809404-120d-4cae-a3b6-cad3e7ccce13'
        ],
        [
            '4ebfdaf8-eb66-43cb-a3a5-9e98ab2b3122',
            'Chunk 4ebfdaf8-eb66-43cb-a3a5-9e98ab2b3122',
            '32809404-120d-4cae-a3b6-cad3e7ccce13'
        ],
        [
            '63e2994d-bf20-4ec0-a396-75eeee383d63',
            'Chunk 63e2994d-bf20-4ec0-a396-75eeee383d63',
            '32809404-120d-4cae-a3b6-cad3e7ccce13'
        ]
    ],
    'entity_count': 57
}

First persisted entities:

{
    'id': '7405115f-d3d0-42bc-9c61-9c9e6b1998cc',
    'name': 'Political-structures case study',
    'labels': ['Concept'],
    'semantic_key': 'concept_case-study'
}

{
    'id': '43807e2c-0064-4e82-8847-394ca50331d3',
    'name': 'Counterfactual reasoning',
    'labels': ['Concept'],
    'semantic_key': 'concept_counterfactual-reasoning'
}

{
    'id': '25a9ffc6-19f8-4e58-b018-7203eb7ff989',
    'name': 'Country',
    'labels': ['Concept'],
    'semantic_key': 'concept_country'
}

...

{
    'id': '45b7ffff-dbb3-4c9a-9d63-773549fb8726',
    'name': 'Graph Neural Networks',
    'labels': ['Concept'],
    'semantic_key': 'concept_gnns'
}

{
    'id': '5b2b6f5e-82f7-4154-a168-5c6d2c01e8e6',
    'name': 'Form of government',
    'labels': ['Concept'],
    'semantic_key': 'concept_gov-type'
}

{
    'id': 'af1eefcb-e0c5-4a9d-86ed-02ef71cdee4d',
    'name': 'Graph databases',
    'labels': ['Concept'],
    'semantic_key': 'concept_graph-databases'
}

{
    'id': '5b658147-5ad1-4ec1-9efc-b34b1154fbed',
    'name': 'Graph Neural Networks',
    'labels': ['Concept'],
    'semantic_key': 'concept_graph-neural-networks'
}

{
    'id': '91766576-1ea8-40fe-b417-82757b382a51',
    'name': 'Hallucination',
    'labels': ['Concept'],
    'semantic_key': 'concept_hallucination'
}

In [5]:
focus_entity = entities[0] if entities else None
if focus_entity is None:
    print("No entities were persisted.")
else:
    neighbors = await database.neighbor_relationships(
        node_ids=[focus_entity.id], limit_per_node=5
    )
    print({"focus_entity": focus_entity.name, "neighbors": neighbors[focus_entity.id]})

search_hits = await rag.search("Knowledge graphs", limit=5)
print("GraphRAG search hits:")
for hit in search_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "id": hit.node.id,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
        }
    )

{
    'focus_entity': 'Political-structures case study',
    'neighbors': [
        NeighborRelationship(
            source_id='7405115f-d3d0-42bc-9c61-9c9e6b1998cc',
            source_name='Political-structures case study',
            relationship_label='__mentions__',
            target=ChunkNode(
                id='03a40b69-0fd4-4a75-9be9-ea9cf16acc09',
                labels=frozenset({'__chunk__'}),
                semantic_key='chunk_03a40b69-0fd4-4a75-9be9-ea9cf16acc09',
                name='Chunk 03a40b69-0fd4-4a75-9be9-ea9cf16acc09',
                properties={},
                embedding=[],
                document_id='32809404-120d-4cae-a3b6-cad3e7ccce13',
                content='You will see, how it can efficiently represent multidimensional data, allowing researchers
and analysts to get useful insights.\n\nIn this case study, we will show how knowledge graphs may effectively 
capture and analyse complicated connections. We will construct a heterogeneous graph that contains selected 
characteristics of political structures and organisation among members of multinational organisations such as NATO 
and the European Union. Can we identify any trends there? Are there any patterns or clusters, such as those, 
connected to the structure of a government or a nation’s constitution?\n\nIn order to build a KG in Neo4j and 
PyTorch Geometric, we will extract specific information about countries, such as their forms of governments, 
legislatures, and political subjects (broader political science categories to which they belong). It will be 
accomplished by utilising DbPedia’s huge and organised data.\n\nThe code for this case study (SPARQL and CYPHER 
queries, as well as Python code) can be found on my Github repository:\n\nQuerying DbPedia\nFirst, let’s start with
the query. DbPedia offers a rich semantic data representation on various topics, e.g., country politics. Below, you
can find a sample record on the European Union (available at: https://dbpedia.org/page/European_Union):\n\nYou can 
think of each record as a dictionary of key-value pairs, where keys are semantic qualifiers of DbPedia ontology 
namespaces while values are links to other records. Therefore, DbPedia data structures are graphs 
themselves!\n\nThe goal of the query is to retrieve the following information:\n\nCountry data, including the name 
and wikiPageID of each nation that belongs to the UN.\nInformation about each nation’s membership in selected 
international organizations, namely: NATO, the European Union, and the Three Seas Initiative.\nDetails about each 
nation’s type of government, including its label, wikiPageID, and related political topics.\nInformation about 
these nations’ legislatures, such as their name and wikiPageID.\n',
                metadata={}
            )
        ),
        NeighborRelationship(
            source_id='7405115f-d3d0-42bc-9c61-9c9e6b1998cc',
            source_name='Political-structures case study',
            relationship_label='sources',
            target=Node(
                id='3889ca88-5aec-43ad-b4ef-71d73b6689cd',
                labels=frozenset({'Dataset'}),
                semantic_key='dataset_dbpedia',
                name='DbPedia',
                properties={},
                embedding=[]
            )
        ),
        NeighborRelationship(
            source_id='7405115f-d3d0-42bc-9c61-9c9e6b1998cc',
            source_name='Political-structures case study',
            relationship_label='code_available_at',
            target=Node(
                id='1520789b-cfc8-44ce-89c0-090d768836ca',
                labels=frozenset({'Resource'}),
                semantic_key='resource_github',
                name='Github repository (case study code)',
                properties={},
                embedding=[]
            )
        ),
        NeighborRelationship(
            source_id='7405115f-d3d0-42bc-9c61-9c9e6b1998cc',
            source_name='Political-structures case study',
        

GraphRAG search hits:

{
    'score': 0.9179,
    'matched_on': 'keyword_path',
    'id': '82cc4a05-871e-414e-a1bc-598830019c7d',
    'name': 'Knowledge Graph',
    'labels': ['Concept']
}

{
    'score': 0.8719,
    'matched_on': 'keyword_path',
    'id': '81f625d8-0edb-417f-ab96-0c95d967e5cd',
    'name': 'Knowledge Graphs (KG)',
    'labels': ['Concept']
}

{
    'score': 0.7248,
    'matched_on': 'keyword_path',
    'id': '3482e1cd-e82f-42f4-b43d-c024b59c0f37',
    'name': 'Open Knowledge Graph',
    'labels': ['Concept']
}

{
    'score': 0.6491,
    'matched_on': 'vector',
    'id': '0f65886d-42fc-4596-932c-f28822e09d32',
    'name': 'Chunk 0f65886d-42fc-4596-932c-f28822e09d32',
    'labels': ['__chunk__']
}

{
    'score': 0.5876,
    'matched_on': 'vector',
    'id': '63e2994d-bf20-4ec0-a396-75eeee383d63',
    'name': 'Chunk 63e2994d-bf20-4ec0-a396-75eeee383d63',
    'labels': ['__chunk__']
}

## Duplicate inspection

Use the duplicate finder first so the notebook shows the candidate groups before applying the destructive merge step. Exact counts may vary slightly across model runs.


In [6]:
def merge_report_payload(report):
    return {
        "master_id": report.master_id,
        "duplicate_ids": list(report.duplicate_ids),
        "source": report.source,
        "merged_labels": list(report.merged_labels),
        "property_conflicts": report.property_conflicts,
    }


duplicate_candidates = await rag.find_entity_duplicate_candidates(
    limit=5, threshold=0.9
)
dry_run_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=True
)

print(
    {
        "semantic_key_collision_groups": len(
            duplicate_candidates.semantic_key_collision_candidates
        ),
        "similarity_candidate_groups": len(duplicate_candidates.similarity_candidates),
        "dry_run_merge_reports": [
            merge_report_payload(report) for report in dry_run_reports
        ],
    }
)

{
    'semantic_key_collision_groups': 0,
    'similarity_candidate_groups': 8,
    'dry_run_merge_reports': [
        {
            'master_id': '14146a27-9e4a-4428-a332-b5a2645244c2',
            'duplicate_ids': ['3889ca88-5aec-43ad-b4ef-71d73b6689cd', '115381d2-6eb5-4d4a-9238-9f425b95ad99'],
            'source': 'similarity',
            'merged_labels': ['DataSource', 'Dataset', 'Project'],
            'property_conflicts': ()
        },
        {
            'master_id': 'ae672091-61b6-403a-9bda-06a8a1d07223',
            'duplicate_ids': ['a4c20c3b-1f90-48b5-b352-43fa49efdedd', 'b4894d47-97e2-4bfa-9227-ed3599adbe89'],
            'source': 'similarity',
            'merged_labels': ['Library', 'Software', 'Tool'],
            'property_conflicts': ()
        },
        {
            'master_id': '4a9613b7-044a-4339-aa41-462f40e9c796',
            'duplicate_ids': [
                '751125c2-5476-43b2-8d0a-c52bc6bd79a9',
                'c1566dd9-3796-4871-84e1-d657dca31339',
                'd74328fe-cbea-41ef-957d-f6b6dd289449'
            ],
            'source': 'similarity',
            'merged_labels': ['Organization', 'Software', 'Tool'],
            'property_conflicts': ()
        }
    ]
}

In [7]:
applied_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=False
)
entities_after_dedupe = await database.list_entities()

print("Applied merge reports:")
for report in applied_reports:
    print(merge_report_payload(report))

print({"entity_count_after_dedupe": len(entities_after_dedupe)})
for entity in entities_after_dedupe[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

Applied merge reports:

{
    'master_id': '14146a27-9e4a-4428-a332-b5a2645244c2',
    'duplicate_ids': ['3889ca88-5aec-43ad-b4ef-71d73b6689cd', '115381d2-6eb5-4d4a-9238-9f425b95ad99'],
    'source': 'similarity',
    'merged_labels': ['DataSource', 'Dataset', 'Project'],
    'property_conflicts': ()
}

{
    'master_id': 'ae672091-61b6-403a-9bda-06a8a1d07223',
    'duplicate_ids': ['a4c20c3b-1f90-48b5-b352-43fa49efdedd', 'b4894d47-97e2-4bfa-9227-ed3599adbe89'],
    'source': 'similarity',
    'merged_labels': ['Library', 'Software', 'Tool'],
    'property_conflicts': ()
}

{
    'master_id': '4a9613b7-044a-4339-aa41-462f40e9c796',
    'duplicate_ids': [
        '751125c2-5476-43b2-8d0a-c52bc6bd79a9',
        'c1566dd9-3796-4871-84e1-d657dca31339',
        'd74328fe-cbea-41ef-957d-f6b6dd289449'
    ],
    'source': 'similarity',
    'merged_labels': ['Organization', 'Software', 'Tool'],
    'property_conflicts': ()
}

{'entity_count_after_dedupe': 50}

...

{
    'id': '45b7ffff-dbb3-4c9a-9d63-773549fb8726',
    'name': 'Graph Neural Networks',
    'labels': ['Concept'],
    'semantic_key': 'concept_gnns'
}

{
    'id': '5b2b6f5e-82f7-4154-a168-5c6d2c01e8e6',
    'name': 'Form of government',
    'labels': ['Concept'],
    'semantic_key': 'concept_gov-type'
}

{
    'id': 'af1eefcb-e0c5-4a9d-86ed-02ef71cdee4d',
    'name': 'Graph databases',
    'labels': ['Concept'],
    'semantic_key': 'concept_graph-databases'
}

{
    'id': '5b658147-5ad1-4ec1-9efc-b34b1154fbed',
    'name': 'Graph Neural Networks',
    'labels': ['Concept'],
    'semantic_key': 'concept_graph-neural-networks'
}

{
    'id': '91766576-1ea8-40fe-b417-82757b382a51',
    'name': 'Hallucination',
    'labels': ['Concept'],
    'semantic_key': 'concept_hallucination'
}

In [10]:
final_hits = await rag.search("Knowledge graph concept", limit=5)
comparison_rows = database.ro_query(
    "MATCH (s:__entity__)-[r]->(t:__entity__) RETURN s.name, type(r), t.name ORDER BY s.name, type(r), t.name LIMIT 10"
).result_set

print("Final GraphRAG hits:")
for hit in final_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
            "content_preview": hit.node.properties.get("content", "")[:500],
        }
    )

print({"entity_relationship_samples": comparison_rows})

Final GraphRAG hits:

{
    'score': 0.8943,
    'matched_on': 'keyword_path',
    'name': 'Knowledge Graph',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Knowledge Graph (id: 82cc4a05-871e-414e-a1bc-598830019c7d), similarity: 
0.8943\n-[__mentions__]-> NAME: Chunk 03a40b69-0fd4-4a75-9be9-ea9cf16acc09; LABELS: __chunk__; CONTENT: You will 
see, how it can efficiently represent multidimensional data, allowing researchers and analysts to get useful 
insights.\n\nIn this case study, we will show how knowledge graphs may effectively capture and analyse complicated 
connections. We will construct a heterogeneous graph that contains selected chara'
}

{
    'score': 0.8122,
    'matched_on': 'keyword_path',
    'name': 'Knowledge Graphs (KG)',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Knowledge Graphs (KG) (id: 81f625d8-0edb-417f-ab96-0c95d967e5cd), similarity: 
0.8122\n-[__mentions__]-> NAME: Chunk 63e2994d-bf20-4ec0-a396-75eeee383d63; LABELS: __chunk__; CONTENT: We have the
following node types: “author”, “paper”, “institution”, and the following relation types: “wrote”, “published”, 
“cooperated”.\n\nI believe that this is much more intuitive than the formal expression above.\n\nAnother variation 
of heterogeneous graphs (or their subtype) is the so-called “property g'
}

{
    'score': 0.7102,
    'matched_on': 'keyword_path',
    'name': 'Open Knowledge Graph',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Open Knowledge Graph (id: 3482e1cd-e82f-42f4-b43d-c024b59c0f37), similarity: 
0.7102\n-[__mentions__]-> NAME: Chunk 0f65886d-42fc-4596-932c-f28822e09d32; LABELS: __chunk__; CONTENT: A knowledge
graph (i) mainly describes real world entities and their interrelations, organized in a graph, (ii) defines 
possible classes and relations of entities in a schema, (iii) allows for potentially interrelating arbitrary 
entities with each other and (iv) covers various topical domains [6].\n\nKnowled'
}

{
    'score': 0.6639,
    'matched_on': 'vector',
    'name': 'Chunk 0f65886d-42fc-4596-932c-f28822e09d32',
    'labels': ['__chunk__'],
    'content_preview': ''
}

{
    'score': 0.597,
    'matched_on': 'vector',
    'name': 'Chunk 63e2994d-bf20-4ec0-a396-75eeee383d63',
    'labels': ['__chunk__'],
    'content_preview': ''
}

{
    'entity_relationship_samples': [
        ['Actors', 'cooperated', 'Directors'],
        ['ChatGPT', 'is_instance_of', 'Large Language Models'],
        ['DbPedia', 'extracts_from', 'Wikimedia sources'],
        ['DbPedia', 'provides', 'Open Knowledge Graph'],
        ['Graph Neural Networks', 'applied_to', 'Indirect Question Answering'],
        ['House', 'part_of', "Nation's legislature"],
        ['Knowledge Graph', 'enrich', 'Large Language Models'],
        ['Large Language Models', 'can_hallucinate', 'Hallucination'],
        ['Large Language Models', 'perform', 'Probabilistic reasoning'],
        ['Movie Graph (Neo4j example dataset)', 'features', 'Actors']
    ]
}

## Next steps

- Notebook 2 reuses the same FalkorDB graph for a `pydantic_ai.Agent` demo with `search`, `remember`, and `recall`.
- Notebook 3 builds a lightweight visualization over the populated graph.
- The other article files in `experimental_data/` are good follow-up ingestion exercises once this baseline flow works.


In [11]:
# Run this once when you are finished with the notebook session.
database.close()